# E-commerce Checkout A/B Test
## SQL Experiment Validation

### Objective

This notebook validates the raw checkout experiment data before any conversion analysis is performed.

The validation focuses on:

- table grain and primary-key integrity;
- duplicate assignment and event records;
- cross-group assignment contamination;
- invalid employee, test-account, and bot traffic;
- checkout events occurring before experiment assignment;
- valid experiment exposure;
- complete 24-hour outcome observation windows; and
- treatment-control balance.

The goal is to construct a trustworthy experiment population before estimating treatment effects.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path(
    r"D:\JNP\analytics 30days July\product-ab-testing-analysis\week2_checkout_experiment"
)

DB_PATH = PROJECT_ROOT / "data" / "processed" / "checkout_experiment.db"

conn = sqlite3.connect(DB_PATH)

print(DB_PATH)

D:\JNP\analytics 30days July\product-ab-testing-analysis\week2_checkout_experiment\data\processed\checkout_experiment.db


In [2]:
pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

,name
0,events
1,experiment_assignments
2,orders
3,users


In [3]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT user_id) AS unique_users
FROM users;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_users
0,20000,20000


In [4]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT assignment_id) AS unique_assignment_ids,
    COUNT(DISTINCT user_id) AS unique_users
FROM experiment_assignments;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_assignment_ids,unique_users
0,20060,20020,20000


In [5]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT event_id) AS unique_event_ids,
    COUNT(DISTINCT user_id) AS unique_users
FROM events;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_event_ids,unique_users
0,36813,36733,17195


In [6]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT user_id) AS unique_purchasers
FROM orders;
"""

pd.read_sql_query(query, conn)

,total_rows,unique_orders,unique_purchasers
0,5043,5043,5043


In [7]:
query = """
SELECT COUNT(*) AS duplicate_assignment_id_count
FROM (
    SELECT assignment_id
    FROM experiment_assignments
    GROUP BY assignment_id
    HAVING COUNT(*) > 1
);
"""

pd.read_sql_query(query, conn)

,duplicate_assignment_id_count
0,40


In [8]:
query = """
SELECT COUNT(*) AS cross_assigned_user_count
FROM (
    SELECT user_id
    FROM experiment_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
);
"""

pd.read_sql_query(query, conn)

,cross_assigned_user_count
0,20


In [9]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS rn
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE rn = 1
),

cross_assigned_users AS (

    SELECT user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
),

deduplicated_events AS (

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_timestamp
            ) AS rn
        FROM events
    )
    WHERE rn = 1
)

SELECT
    COUNT(*) AS checkout_events_before_assignment
FROM canonical_assignments a
JOIN deduplicated_events e
    ON a.user_id = e.user_id
WHERE e.event_name = 'checkout_view'
  AND e.event_timestamp < a.assignment_timestamp;
"""

pd.read_sql_query(query, conn)

,checkout_events_before_assignment
0,30


In [10]:
query = """
SELECT
    COUNT(*) AS total_users,
    SUM(CASE WHEN is_employee = 1 THEN 1 ELSE 0 END) AS employees,
    SUM(CASE WHEN is_test_account = 1 THEN 1 ELSE 0 END) AS test_accounts,
    SUM(CASE WHEN is_bot = 1 THEN 1 ELSE 0 END) AS bots,
    SUM(
        CASE
            WHEN is_employee = 0
             AND is_test_account = 0
             AND is_bot = 0
            THEN 1 ELSE 0
        END
    ) AS eligible_users
FROM users;
"""

pd.read_sql_query(query, conn)

,total_users,employees,test_accounts,bots,eligible_users
0,20000,51,43,116,19791


In [11]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS rn
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE rn = 1
),

cross_assigned_users AS (

    SELECT user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
),

eligible_users AS (

    SELECT *
    FROM users
    WHERE is_employee = 0
      AND is_test_account = 0
      AND is_bot = 0
),

deduplicated_events AS (

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_timestamp
            ) AS rn
        FROM events
    )
    WHERE rn = 1
),

valid_checkout_events AS (

    SELECT
        a.user_id,
        a.experiment_group,
        a.assignment_timestamp,
        e.event_timestamp,
        e.device,
        e.traffic_source
    FROM canonical_assignments a

    JOIN eligible_users u
        ON a.user_id = u.user_id

    JOIN deduplicated_events e
        ON a.user_id = e.user_id

    WHERE e.event_name = 'checkout_view'
      AND e.event_timestamp >= a.assignment_timestamp
      AND e.experiment_group = a.experiment_group
),

first_valid_exposure AS (

    SELECT
        user_id,
        experiment_group,
        MIN(event_timestamp) AS exposure_timestamp
    FROM valid_checkout_events
    GROUP BY
        user_id,
        experiment_group
)

SELECT
    COUNT(*) AS exposed_users
FROM first_valid_exposure;
"""

pd.read_sql_query(query, conn)

,exposed_users
0,16992


In [12]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS rn
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE rn = 1
),

cross_assigned_users AS (

    SELECT user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
),

eligible_users AS (

    SELECT *
    FROM users
    WHERE is_employee = 0
      AND is_test_account = 0
      AND is_bot = 0
),

deduplicated_events AS (

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_timestamp
            ) AS rn
        FROM events
    )
    WHERE rn = 1
),

valid_checkout_events AS (

    SELECT
        a.user_id,
        a.experiment_group,
        a.assignment_timestamp,
        e.event_timestamp,
        e.device,
        e.traffic_source
    FROM canonical_assignments a

    JOIN eligible_users u
        ON a.user_id = u.user_id

    JOIN deduplicated_events e
        ON a.user_id = e.user_id

    WHERE e.event_name = 'checkout_view'
      AND e.event_timestamp >= a.assignment_timestamp
      AND e.experiment_group = a.experiment_group
),

ranked_exposures AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_timestamp
        ) AS exposure_rank
    FROM valid_checkout_events
),

first_valid_exposure AS (

    SELECT
        user_id,
        experiment_group,
        assignment_timestamp,
        event_timestamp AS exposure_timestamp,
        device AS device_at_exposure,
        traffic_source AS traffic_source_at_exposure
    FROM ranked_exposures
    WHERE exposure_rank = 1
),

mature_experiment_population AS (

    SELECT *
    FROM first_valid_exposure
    WHERE datetime(
        exposure_timestamp,
        '+24 hours'
    ) <= datetime('2026-07-15 12:00:00')
)

SELECT
    COUNT(*) AS mature_exposed_users
FROM mature_experiment_population;
"""

pd.read_sql_query(query, conn)

,mature_exposed_users
0,15467


In [13]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS rn
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE rn = 1
),

cross_assigned_users AS (

    SELECT user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
),

eligible_users AS (

    SELECT *
    FROM users
    WHERE is_employee = 0
      AND is_test_account = 0
      AND is_bot = 0
),

deduplicated_events AS (

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_timestamp
            ) AS rn
        FROM events
    )
    WHERE rn = 1
),

valid_checkout_events AS (

    SELECT
        a.user_id,
        a.experiment_group,
        a.assignment_timestamp,
        e.event_timestamp,
        e.device,
        e.traffic_source
    FROM canonical_assignments a

    JOIN eligible_users u
        ON a.user_id = u.user_id

    JOIN deduplicated_events e
        ON a.user_id = e.user_id

    WHERE e.event_name = 'checkout_view'
      AND e.event_timestamp >= a.assignment_timestamp
      AND e.experiment_group = a.experiment_group
),

ranked_exposures AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_timestamp
        ) AS exposure_rank
    FROM valid_checkout_events
),

first_valid_exposure AS (

    SELECT
        user_id,
        experiment_group,
        assignment_timestamp,
        event_timestamp AS exposure_timestamp,
        device AS device_at_exposure,
        traffic_source AS traffic_source_at_exposure
    FROM ranked_exposures
    WHERE exposure_rank = 1
),

mature_experiment_population AS (

    SELECT *
    FROM first_valid_exposure
    WHERE datetime(
        exposure_timestamp,
        '+24 hours'
    ) <= datetime('2026-07-15 12:00:00')
)

SELECT
    experiment_group,
    COUNT(*) AS users,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM mature_experiment_population
GROUP BY experiment_group;
"""

pd.read_sql_query(query, conn)

,experiment_group,users,percentage
0,control,7754,50.13
1,treatment,7713,49.87


In [14]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS rn
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE rn = 1
),

cross_assigned_users AS (

    SELECT user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
),

eligible_users AS (

    SELECT *
    FROM users
    WHERE is_employee = 0
      AND is_test_account = 0
      AND is_bot = 0
),

deduplicated_events AS (

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_timestamp
            ) AS rn
        FROM events
    )
    WHERE rn = 1
),

valid_checkout_events AS (

    SELECT
        a.user_id,
        a.experiment_group,
        a.assignment_timestamp,
        e.event_timestamp,
        e.device,
        e.traffic_source
    FROM canonical_assignments a

    JOIN eligible_users u
        ON a.user_id = u.user_id

    JOIN deduplicated_events e
        ON a.user_id = e.user_id

    WHERE e.event_name = 'checkout_view'
      AND e.event_timestamp >= a.assignment_timestamp
      AND e.experiment_group = a.experiment_group
),

ranked_exposures AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_timestamp
        ) AS exposure_rank
    FROM valid_checkout_events
),

first_valid_exposure AS (

    SELECT
        user_id,
        experiment_group,
        assignment_timestamp,
        event_timestamp AS exposure_timestamp,
        device AS device_at_exposure,
        traffic_source AS traffic_source_at_exposure
    FROM ranked_exposures
    WHERE exposure_rank = 1
),

mature_experiment_population AS (

    SELECT *
    FROM first_valid_exposure
    WHERE datetime(
        exposure_timestamp,
        '+24 hours'
    ) <= datetime('2026-07-15 12:00:00')
)

SELECT
    p.experiment_group,
    u.user_type,
    COUNT(*) AS users,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (
            PARTITION BY p.experiment_group
        ),
        2
    ) AS pct_within_group
FROM mature_experiment_population p
JOIN users u
    ON p.user_id = u.user_id
GROUP BY
    p.experiment_group,
    u.user_type
ORDER BY
    p.experiment_group,
    u.user_type;
"""

pd.read_sql_query(query, conn)

,experiment_group,user_type,users,pct_within_group
0,control,new,2371,30.58
1,control,returning,5383,69.42
2,treatment,new,2292,29.72
3,treatment,returning,5421,70.28


In [15]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS rn
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE rn = 1
),

cross_assigned_users AS (

    SELECT user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
),

eligible_users AS (

    SELECT *
    FROM users
    WHERE is_employee = 0
      AND is_test_account = 0
      AND is_bot = 0
),

deduplicated_events AS (

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_timestamp
            ) AS rn
        FROM events
    )
    WHERE rn = 1
),

valid_checkout_events AS (

    SELECT
        a.user_id,
        a.experiment_group,
        a.assignment_timestamp,
        e.event_timestamp,
        e.device,
        e.traffic_source
    FROM canonical_assignments a

    JOIN eligible_users u
        ON a.user_id = u.user_id

    JOIN deduplicated_events e
        ON a.user_id = e.user_id

    WHERE e.event_name = 'checkout_view'
      AND e.event_timestamp >= a.assignment_timestamp
      AND e.experiment_group = a.experiment_group
),

ranked_exposures AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_timestamp
        ) AS exposure_rank
    FROM valid_checkout_events
),

first_valid_exposure AS (

    SELECT
        user_id,
        experiment_group,
        assignment_timestamp,
        event_timestamp AS exposure_timestamp,
        device AS device_at_exposure,
        traffic_source AS traffic_source_at_exposure
    FROM ranked_exposures
    WHERE exposure_rank = 1
),

mature_experiment_population AS (

    SELECT *
    FROM first_valid_exposure
    WHERE datetime(
        exposure_timestamp,
        '+24 hours'
    ) <= datetime('2026-07-15 12:00:00')
)

SELECT
    experiment_group,
    device_at_exposure,
    COUNT(*) AS users,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (
            PARTITION BY experiment_group
        ),
        2
    ) AS pct_within_group
FROM mature_experiment_population
GROUP BY
    experiment_group,
    device_at_exposure
ORDER BY
    experiment_group,
    device_at_exposure;
"""

pd.read_sql_query(query, conn)

,experiment_group,device_at_exposure,users,pct_within_group
0,control,desktop,2805,36.17
1,control,mobile,4461,57.53
2,control,tablet,488,6.29
3,treatment,desktop,2785,36.11
4,treatment,mobile,4469,57.94
5,treatment,tablet,459,5.95


In [16]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS rn
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE rn = 1
),

cross_assigned_users AS (

    SELECT user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
),

eligible_users AS (

    SELECT *
    FROM users
    WHERE is_employee = 0
      AND is_test_account = 0
      AND is_bot = 0
),

deduplicated_events AS (

    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY event_timestamp
            ) AS rn
        FROM events
    )
    WHERE rn = 1
),

valid_checkout_events AS (

    SELECT
        a.user_id,
        a.experiment_group,
        a.assignment_timestamp,
        e.event_timestamp,
        e.device,
        e.traffic_source
    FROM canonical_assignments a

    JOIN eligible_users u
        ON a.user_id = u.user_id

    JOIN deduplicated_events e
        ON a.user_id = e.user_id

    WHERE e.event_name = 'checkout_view'
      AND e.event_timestamp >= a.assignment_timestamp
      AND e.experiment_group = a.experiment_group
),

ranked_exposures AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY event_timestamp
        ) AS exposure_rank
    FROM valid_checkout_events
),

first_valid_exposure AS (

    SELECT
        user_id,
        experiment_group,
        assignment_timestamp,
        event_timestamp AS exposure_timestamp,
        device AS device_at_exposure,
        traffic_source AS traffic_source_at_exposure
    FROM ranked_exposures
    WHERE exposure_rank = 1
),

mature_experiment_population AS (

    SELECT *
    FROM first_valid_exposure
    WHERE datetime(
        exposure_timestamp,
        '+24 hours'
    ) <= datetime('2026-07-15 12:00:00')
)

SELECT
    experiment_group,
    traffic_source_at_exposure,
    COUNT(*) AS users,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (
            PARTITION BY experiment_group
        ),
        2
    ) AS pct_within_group
FROM mature_experiment_population
GROUP BY
    experiment_group,
    traffic_source_at_exposure
ORDER BY
    experiment_group,
    traffic_source_at_exposure;
"""

pd.read_sql_query(query, conn)

,experiment_group,traffic_source_at_exposure,users,pct_within_group
0,control,direct,1282,16.53
1,control,email,1212,15.63
2,control,organic,2374,30.62
3,control,paid_search,1929,24.88
4,control,social,957,12.34
5,treatment,direct,1332,17.27
6,treatment,email,1231,15.96
7,treatment,organic,2327,30.17
8,treatment,paid_search,1909,24.75
9,treatment,social,914,11.85


## Validation Summary

The raw experiment data contained the intentional quality issues introduced during simulation, including duplicate assignments, cross-group assignments, duplicate events, and checkout events occurring before assignment.

A temporal consistency check confirmed that no canonical experiment assignment occurred before the user's first observed timestamp.

After removing invalid assignment records, excluding internal/test/bot traffic, identifying the first valid checkout exposure, and requiring a complete 24-hour conversion observation window, the final mature analysis population contained 15,467 users.

The final sample was approximately evenly split between control and treatment:

- Control: 7,754 users (50.13%)
- Treatment: 7,713 users (49.87%)

User type, device-at-exposure, and traffic-source distributions were broadly similar across experiment groups, with no substantial treatment-control imbalance observed.

The validated population is therefore suitable for downstream funnel and treatment-effect analysis.

In [17]:
query = """
WITH ranked_assignments AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY assignment_id
            ORDER BY assignment_timestamp
        ) AS assignment_row_number
    FROM experiment_assignments
),

deduplicated_assignments AS (

    SELECT
        assignment_id,
        experiment_name,
        user_id,
        experiment_group,
        assignment_timestamp
    FROM ranked_assignments
    WHERE assignment_row_number = 1
),

cross_assigned_users AS (

    SELECT
        user_id
    FROM deduplicated_assignments
    GROUP BY user_id
    HAVING COUNT(DISTINCT experiment_group) > 1
),

canonical_assignments AS (

    SELECT da.*
    FROM deduplicated_assignments da
    LEFT JOIN cross_assigned_users ca
        ON da.user_id = ca.user_id
    WHERE ca.user_id IS NULL
)

SELECT
    COUNT(*) AS assignment_before_first_seen
FROM canonical_assignments a
JOIN users u
    ON a.user_id = u.user_id
WHERE datetime(a.assignment_timestamp)
    < datetime(u.first_seen_timestamp);
"""

pd.read_sql_query(query, conn)

,assignment_before_first_seen
0,0
